多个中间件组合及执行顺序

In [1]:
from typing import Any
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware
from langchain.agents.middleware.types import StateT
from langgraph.runtime import Runtime
from langgraph.typing import ContextT
from dotenv import load_dotenv
import os

load_dotenv(override=True)

main_model = init_chat_model(
    model="qwen3.7-flash",
    model_provider="openai",
    api_key = os.getenv("DASHSCOPE_API_KEY"),
    base_url = os.getenv("DASHSCOPE_BASE_URL"),
)

class Middleware1(AgentMiddleware):
    def before_model(self, state: StateT, runtime: Runtime[ContextT]) -> dict[str, Any] | None:
        print("【中间件1】before_model")
        return None
    def after_model(self, state: StateT, runtime: Runtime[ContextT]) -> dict[str, Any] | None:
        print("【中间件1】after_model")
        return None

class Middleware2(AgentMiddleware):
    def before_model(self, state: StateT, runtime: Runtime[ContextT]) -> dict[str, Any] | None:
        print("【中间件2】before_model")
        return None
    def after_model(self, state: StateT, runtime: Runtime[ContextT]) -> dict[str, Any] | None:
        print("【中间件2】after_model")
        return None

class Middleware3(AgentMiddleware):
    def before_model(self, state: StateT, runtime: Runtime[ContextT]) -> dict[str, Any] | None:
        print("【中间件3】before_model")
        return None
    def after_model(self, state: StateT, runtime: Runtime[ContextT]) -> dict[str, Any] | None:
        print("【中间件3】after_model")
        return None

agent = create_agent(
    model=main_model,
    tools=[],
    middleware=[Middleware1(), Middleware2(), Middleware3()],
)

print("\n执行一次调用，观察顺序：")
agent.invoke({
    "messages": [
        {"role": "user", "content": "测试"}
    ]
})

print("\n关键点：")
print(" - before_model: 正序执行（1→2→3）")
print(" - after_model: 逆序执行（3→2→1）")
print(" - 类似洋葱模型：1→2→3→模型→3→2→1")


执行一次调用，观察顺序：
【中间件1】before_model
【中间件2】before_model
【中间件3】before_model
【中间件3】after_model
【中间件2】after_model
【中间件1】after_model

关键点：
 - before_model: 正序执行（1→2→3）
 - after_model: 逆序执行（3→2→1）
 - 类似洋葱模型：1→2→3→模型→3→2→1
